#Importación de librerarias

In [34]:
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler


#Carga y preprocesamiento de datos

In [35]:
# --------------------------------------------------------------------------
# 1. CONFIGURACIÓN
# --------------------------------------------------------------------------
DATA_PATH = "https://raw.githubusercontent.com/No-Country-simulation/Hackaton_G9_Server_2_Team27/refs/heads/feature/ds-notebooks-modelos/data/processed/dataset_ml_ready_v2.csv"
TARGET_COLUMN = "categoria"
MODEL_OUTPUT_PATH = "modelo_knn_pipeline.pkl"
RANDOM_STATE = 42
TEST_SIZE = 0.2

# Columnas numéricas continuas/discretas -> se escalan con RobustScaler
# (robusto a outliers; consumo_kwh tiene una cola larga: max=1747 vs media=479)
NUMERIC_FEATURES = ["consumo_kwh", "cantidad_equipos", "horas_alto_consumo"]

# Columnas categóricas nominales -> se codifican con One-Hot
CATEGORICAL_FEATURES = ["tipo_vivienda"]

# Columna booleana -> ya es 0/1, se deja pasar tal cual (passthrough)
BOOLEAN_FEATURES = ["uso_horario_pico"]


def build_preprocessor() -> ColumnTransformer:
    """
    Arma el ColumnTransformer que centraliza TODO el preprocesamiento.
    Al vivir dentro del Pipeline final, se entrena (fit) únicamente sobre
    X_train y se aplica (transform) de forma idéntica en entrenamiento,
    validación y producción (FastAPI).
    """
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", RobustScaler(), NUMERIC_FEATURES),
            (
                "cat",
                OneHotEncoder(handle_unknown="ignore", drop="if_binary"),
                CATEGORICAL_FEATURES,
            ),
            ("bool", "passthrough", BOOLEAN_FEATURES),
        ],
        remainder="drop",  # cualquier columna no listada se descarta explícitamente
        verbose_feature_names_out=False,
    )
    return preprocessor


def build_pipeline() -> Pipeline:
    """
    Pipeline único: preprocesamiento + modelo.
    Este es el objeto que se serializa y se carga en FastAPI.
    """
    pipeline = Pipeline(
        steps=[
            ("preprocessor", build_preprocessor()),
            ("classifier", KNeighborsClassifier()),
        ]
    )
    return pipeline




#Entrenamiento

In [36]:
def main():
    # ----------------------------------------------------------------------
    # 2. CARGA DE DATOS
    # ----------------------------------------------------------------------
    df = pd.read_csv(DATA_PATH)

    # uso_horario_pico llega como bool de pandas -> se castea a int para
    # evitar problemas de tipado al deserializar JSON en FastAPI (los bools
    # de Python/JSON se mapean bien, pero se normaliza por consistencia).
    df["uso_horario_pico"] = df["uso_horario_pico"].astype(int)

    feature_cols = NUMERIC_FEATURES + CATEGORICAL_FEATURES + BOOLEAN_FEATURES
    X = df[feature_cols]
    y = df[TARGET_COLUMN]

    # ----------------------------------------------------------------------
    # 3. SPLIT TRAIN / TEST
    # ----------------------------------------------------------------------
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )

    # ----------------------------------------------------------------------
    # 4. PIPELINE + BÚSQUEDA DE HIPERPARÁMETROS
    # ----------------------------------------------------------------------
    pipeline = build_pipeline()

    param_grid = {
        "classifier__n_neighbors": [3, 5, 7, 9, 11, 15],
        "classifier__weights": ["uniform", "distance"],
        "classifier__metric": ["minkowski", "manhattan"],
    }

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=5,
        scoring="f1_macro",
        n_jobs=-1,
    )

    # fit se llama SIEMPRE sobre el Pipeline completo (preprocesamiento
    # incluido). El ColumnTransformer se ajusta únicamente con X_train,
    # sin fuga de información del set de test.
    grid_search.fit(X_train, y_train)

    pipeline = grid_search.best_estimator_
    print(f"Mejores hiperparámetros: {grid_search.best_params_}")

    # ----------------------------------------------------------------------
    # 5. EVALUACIÓN
    # ----------------------------------------------------------------------
    y_pred = pipeline.predict(X_test)

    print(f"\nAccuracy en test: {accuracy_score(y_test, y_pred):.4f}")
    print("\nReporte de clasificación:")
    print(classification_report(y_test, y_pred))
    print("Matriz de confusión:")
    print(confusion_matrix(y_test, y_pred))


     # ----------------------------------------------------------------------
    # 6. EXPORTACIÓN DEL PIPELINE COMPLETO (un único .pkl)
    # ----------------------------------------------------------------------
    # Se guarda el pipeline COMPLETO (ColumnTransformer + KNN ya entrenados).
    # En FastAPI basta con:
    #     pipeline = joblib.load("modelo_knn_pipeline.pkl")
    #     pipeline.predict(nuevo_dataframe)
    joblib.dump(pipeline, MODEL_OUTPUT_PATH)
    print(f"\nPipeline exportado correctamente en: {MODEL_OUTPUT_PATH}")

    return pipeline, X_test, y_test


#Resultados

In [37]:
if __name__ == "__main__":
  main()


Mejores hiperparámetros: {'classifier__metric': 'minkowski', 'classifier__n_neighbors': 15, 'classifier__weights': 'distance'}

Accuracy en test: 0.8919

Reporte de clasificación:
              precision    recall  f1-score   support

           0       0.92      0.89      0.90       655
           1       0.81      0.88      0.84       657
           2       0.96      0.91      0.93       668

    accuracy                           0.89      1980
   macro avg       0.90      0.89      0.89      1980
weighted avg       0.90      0.89      0.89      1980

Matriz de confusión:
[[581  73   1]
 [ 51 580  26]
 [  0  63 605]]

Pipeline exportado correctamente en: modelo_knn_pipeline.pkl


#Conclusión del Modelo

El modelo KNN (K-Nearest Neighbors) integrado dentro del Pipeline alcanzó una exactitud (Accuracy) de 89.19% en el conjunto de prueba (1,980 instancias).  
La métrica F1-score Macro es de 0.89, lo que demuestra un rendimiento sólido y equilibrado entre todas las categorías, evitando sesgos hacia clases particulares.  

Análisis por Clase (Categorías de Consumo):

Clase 0 (Eficiente): Obtiene una precisión de 0.92 y un recall de 0.89 (F1-score: 0.90).  

Clase 1 (Moderado): Registra una precisión de 0.81 y un recall de 0.88 (F1-score: 0.84). Es la clase con mayor índice de falsos positivos (73 confusiones con Clase 0 y 63 con Clase 2).  

Clase 2 (Ineficiente): Presenta la precisión más alta con 0.96 y un recall de 0.91 (F1-score: 0.93).

Casi no presenta confusión con la Clase 0 (solo 1 caso), lo cual indica una clara separación entre los niveles extremos de consumo.  

Optimizaciones Clave:

La búsqueda de hiperparámetros (GridSearchCV) determinó que la mejor configuración utiliza n_neighbors=15, ponderación por distance y métrica minkowski. La ponderación por distancia favorece a las observaciones más cercanas, mitigando el impacto del ruido.  

El uso de RobustScaler fue clave para tratar las variables numéricas (especialmente consumo_kwh), reduciendo el impacto negativo de valores atípicos y colas largas.  